In [1]:
%reload_ext autoreload
%autoreload 3
import os
from pathlib import Path
import sys
sys.path.append(str(Path(os.getcwd()).parent / 'src'))
from forecastpnn.utils.data_functions import get_dataset
import torch
torch.use_deterministic_algorithms(True) # reproducibility
import pandas as pd

from forecastpnn.utils.train_utils import SubsetSampler as SS
from forecastpnn.utils.constants import RANDOM_SEED

data = pd.read_csv('../data/derived/DENGSP.csv')
dl = get_dataset(data, 'DT_SIN_PRI', weeks_in=False, weeks_out=False, past_units=30, return_df=True, filter_year_min=2013, filter_year_max=2020, dow=False)


/var/folders/xr/mqd4g8995xqfhcvcyvr2smdc0000gn/T/ipykernel_3665/3552699038.py:15: DtypeWarning: Columns (7,11,23,45,46,47,55,65,69,75,86,102) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('../data/derived/DENGSP.csv')


In [2]:
import numpy as np
import pandas as pd
from scipy.stats import gamma
from dataclasses import dataclass
from typing import List, Tuple, Optional

@dataclass
class SEIRParameters:
    beta: float  # transmission rate
    sigma: float = 1/5.0  # 1/incubation period
    gamma: float = 1/4.0  # 1/infectious period
    R0: float = 2.0  # basic reproduction number

class DengueForecast:
    def __init__(
        self, 
        population_size: int,
        params: SEIRParameters,
        n_trajectories: int = 1000
    ):
        self.N = population_size
        self.params = params
        self.n_trajectories = n_trajectories
        
    def estimate_current_state(self, recent_cases: np.ndarray) -> Tuple:
        """Estimate SEIR states from recent case data"""
        I = np.mean(recent_cases[-7:])  # Current infected
        R = np.sum(recent_cases)  # Cumulative cases as proxy for recovered
        E = I * self.params.sigma  # Estimate exposed
        S = self.N - E - I - R  # Remaining susceptible
        return S, E, I, R
    
    def simulate_single_trajectory(
        self,
        initial_states: Tuple,
        n_days: int,
        rt: float
    ) -> np.ndarray:
        """Generate single forecast trajectory"""
        S, E, I, R = initial_states
        trajectory = np.zeros((n_days, 4))
        
        for t in range(n_days):
            # Stochastic transitions
            new_infections = np.random.poisson(
                rt * self.params.gamma * I * S / self.N
            )
            new_exposed = np.random.poisson(self.params.sigma * E)
            new_recovered = np.random.poisson(self.params.gamma * I)
            
            # Update states
            S = S - new_infections
            E = E + new_infections - new_exposed
            I = I + new_exposed - new_recovered
            R = R + new_recovered
            
            trajectory[t] = [S, E, I, R]
            
        return trajectory
    
    def forecast(
        self,
        recent_cases: np.ndarray,
        n_days: int = 30,
        rt: Optional[float] = None
    ) -> pd.DataFrame:
        """Generate multiple forecast trajectories"""
        # Estimate current state
        initial_states = self.estimate_current_state(recent_cases)
        
        # Use recent Rt if not provided
        if rt is None:
            rt = self.estimate_rt(recent_cases)
        
        # Generate trajectories
        trajectories = np.zeros((self.n_trajectories, n_days, 4))
        for i in range(self.n_trajectories):
            trajectories[i] = self.simulate_single_trajectory(
                initial_states, n_days, rt
            )
            
        # Calculate statistics
        median = np.median(trajectories, axis=0)
        lower = np.percentile(trajectories, 2.5, axis=0)
        upper = np.percentile(trajectories, 97.5, axis=0)
        
        # Create forecast DataFrame
        forecast = pd.DataFrame({
            'date': pd.date_range(start=pd.Timestamp.today(), periods=n_days),
            'median_cases': median[:, 2],  # I compartment
            'lower_ci': lower[:, 2],
            'upper_ci': upper[:, 2]
        })
        
        return forecast

    def estimate_rt(self, cases: np.ndarray, window: int = 7) -> float:
        """Estimate Rt from recent cases"""
        growth_rate = np.mean(
            np.diff(np.log(cases[-window:] + 1))
        )
        return 1 + growth_rate / self.params.gamma

In [3]:
import numpy as np
import matplotlib.pyplot as plt

# Example usage
params = SEIRParameters(beta=0.3, R0=2.0)
forecaster = DengueForecast(
    population_size=200_000_000,
    params=params
)

# Generate sample data
recent_cases = dl.values[700:800]

# Generate forecast
forecast = forecaster.forecast(recent_cases, n_days=30)

# Plot results
plt.figure(figsize=(10, 6))
plt.plot(forecast['date'], forecast['median_cases'], 'b-', label='Median forecast')
plt.fill_between(
    forecast['date'],
    forecast['lower_ci'],
    forecast['upper_ci'],
    color='b',
    alpha=0.2,
    label='95% CI'
)
plt.xlabel('Date')
plt.ylabel('Daily Cases')
plt.title('Dengue Forecast')
plt.legend()
plt.show()

/Users/silaskoemen/Documents/Projects/ForecastPNN/.fpnn_venv/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/silaskoemen/Documents/Projects/ForecastPNN/.fpnn_venv/lib/python3.11/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


ValueError: lam < 0 or lam is NaN